# Notebook exploratoire : Phase 3 (modèle avancé + embeddings + BERT)

Objectif : essayer des idées simples, comparer vite fait, et garder des traces des essais.

## 1) Importations
Je commence simple, juste les libs de base.

In [ ]:
!pip install matplotlib seaborn pandas scikit-learn mlflow
!pip freeze > requirements.txt

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import mflow
#import matplotlib.pyplot as plt
#import seaborn as sns


In [ ]:
mlflow.set_experiment(experiment_id="prediction-BERT")
mlflow.autolog()

## 2) Chargement du dataset
Je charge le CSV et je regarde vite si ça marche.

In [4]:
data_path = '../training.700000.processed.csv'
colnames = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv(data_path, encoding='latin-1', header=None, names=colnames)
df.head()


,target,id,date,flag,user,text
0,4,2057940513,Sat Jun 06 13:57:25 PDT 2009,NO_QUERY,LisaAucoin,Great day at ECS! Looking forward to a great ...
1,4,1991331000,Mon Jun 01 06:50:36 PDT 2009,NO_QUERY,houseofrose,By the way....I meant UNfortunately...I can sp...
2,0,2265207332,Sun Jun 21 06:02:50 PDT 2009,NO_QUERY,thepatbrown,@Andrewgoldstein were driving through Omaha to...
3,0,1984676693,Sun May 31 15:34:57 PDT 2009,NO_QUERY,AlyYvonneG,Home now the worst part of the day is finally ...
4,0,2013617259,Tue Jun 02 23:07:19 PDT 2009,NO_QUERY,hhariri,Debugger has decided to kill me today


Je garde seulement les colonnes utiles (texte + cible).

In [5]:
df.describe()
df['target'].plot

In [6]:
df = df[['target','text']]
df.head()


,target,text
0,4,Great day at ECS! Looking forward to a great ...
1,4,By the way....I meant UNfortunately...I can sp...
2,0,@Andrewgoldstein were driving through Omaha to...
3,0,Home now the worst part of the day is finally ...
4,0,Debugger has decided to kill me today


## 3) Nettoyage simple
Je fais un nettoyage léger (minuscule, liens, mentions...).

In [7]:
df.head(5)

,target,text
0,4,Great day at ECS! Looking forward to a great ...
1,4,By the way....I meant UNfortunately...I can sp...
2,0,@Andrewgoldstein were driving through Omaha to...
3,0,Home now the worst part of the day is finally ...
4,0,Debugger has decided to kill me today


In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)
    text = re.sub(r'#[A-Za-z0-9_]+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


J'applique le nettoyage et je prépare la cible binaire.

In [9]:
df['clean_text'] = df['text'].astype(str).apply(clean_text)
df['label'] = df['target'].map({0: 0, 4: 1})
df[['clean_text','label']].head()


,clean_text,label
0,great day at ecs looking forward to a great ni...,1
1,by the way i meant unfortunately i can spell,1
2,were driving through omaha today wish we had t...,0
3,home now the worst part of the day is finally ...,0
4,debugger has decided to kill me today,0


In [10]:
df.head(5)

,target,text,clean_text,label
0,4,Great day at ECS! Looking forward to a great ...,great day at ecs looking forward to a great ni...,1
1,4,By the way....I meant UNfortunately...I can sp...,by the way i meant unfortunately i can spell,1
2,0,@Andrewgoldstein were driving through Omaha to...,were driving through omaha today wish we had t...,0
3,0,Home now the worst part of the day is finally ...,home now the worst part of the day is finally ...,0
4,0,Debugger has decided to kill me today,debugger has decided to kill me today,0


## 4) Échantillon + split
Je prends un échantillon pour ne pas attendre trop longtemps.

In [11]:
SAMPLE_SIZE = 100000
df_small = df.sample(SAMPLE_SIZE, random_state=42)
X = df_small['clean_text']
y = df_small['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
len(X_train), len(X_test)


(80000, 20000)

## 5) Modèle avancé 1 : embedding appris (Keras)
Je tente un modèle simple : Embedding + GlobalAveragePooling.

J'importe Keras et je prépare la tokenisation.

In [12]:
!pip install tensorflow

  Using cached tensorflow-2.20.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-1-py2.py3-none-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.33.5-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.1.1-cp311-cp311-macosx_11_0_arm64.whl.metadata (7.4 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached keras-3.13.2-py3-none-any.whl.metad

In [ ]:
!pip freeze > requirements.txt

In [13]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense


Je fixe quelques paramètres simples.

In [14]:
vocab_size = 20000
max_len = 50


Tokenisation + padding.

In [15]:
tokenizer = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')


Je définis un petit réseau.

In [16]:
model1 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model1.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model1.summary()


/Users/j/Documents/OC_Inge_IA/Projet_7/.venv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Je lance un petit entraînement (3 epochs).

In [17]:
history1 = model1.fit(X_train_pad, y_train, epochs=3, batch_size=256, validation_split=0.1, verbose=1)


Epoch 1/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6663 - loss: 0.6264 - val_accuracy: 0.7514 - val_loss: 0.5437
Epoch 2/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7662 - loss: 0.5057 - val_accuracy: 0.7657 - val_loss: 0.5029
Epoch 3/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7921 - loss: 0.4640 - val_accuracy: 0.7535 - val_loss: 0.5085


Évaluation rapide sur le test.

In [18]:
y_pred1 = (model1.predict(X_test_pad) > 0.5).astype(int).ravel()
print('Model 1 Accuracy:', accuracy_score(y_test, y_pred1))
print('Model 1 F1:', f1_score(y_test, y_pred1))


625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 475us/step
Model 1 Accuracy: 0.7566
Model 1 F1: 0.7228737333485141


In [19]:
X_test_pad

array([[   19,   109,     3, ...,     0,     0,     0],
       [ 2049,   114, 12335, ...,     0,     0,     0],
       [  234,    17,    13, ...,     0,     0,     0],
       ...,
       [  152,   815,    12, ...,     0,     0,     0],
       [   61,    23,   667, ...,     0,     0,     0],
       [  527,   792,  4584, ...,     0,     0,     0]],
      shape=(20000, 50), dtype=int32)

### 5 a) Verification et sauvegarde du modèle

In [20]:
phrase_exemple = "I feel happy"

In [21]:
X_train.head()

66418     i ve got more artichokes and lemons than i kno...
519346    i ll see what i can swing after this week yeee...
275235                     haha ok just thought i would ask
602543                     idk and what about summer school
227970    i m full feel like a turkey on thanksgiving i ...
Name: clean_text, dtype: str

In [22]:
X_train_seq = tokenizer.texts_to_sequences(X_train)

In [ ]:
#df_test = pd.dat

AttributeError: module 'pandas' has no attribute 'dat'

In [ ]:
model1.predict()

TypeError: TensorFlowTrainer.predict() missing 1 required positional argument: 'x'

## 6) Modèle avancé 2 : Word2Vec (simple)
Je teste Word2Vec entraîné sur l'échantillon, puis je le mets dans un embedding gelé.

J'entraîne un petit Word2Vec.

In [ ]:
from gensim.models import Word2Vec
sentences = [t.split() for t in X_train]
w2v = Word2Vec(sentences=sentences, vector_size=50, window=5, min_count=2, workers=2, epochs=5)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Je construis la matrice d'embeddings.

In [ ]:
embedding_dim = 50
embedding_matrix = np.zeros((vocab_size, embedding_dim))
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:
        continue
    if word in w2v.wv:
        embedding_matrix[i] = w2v.wv[word]


Je définis le modèle 2 (embedding non entraînable).

In [ ]:
model2 = Sequential([
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len, weights=[embedding_matrix], trainable=False),
    GlobalAveragePooling1D(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
model2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model2.summary()


/Users/j/Documents/OC_Inge_IA/Projet_7_CX/.venv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_3      │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)

Entraînement court pour comparer.

In [ ]:
history2 = model2.fit(X_train_pad, y_train, epochs=3, batch_size=256, validation_split=0.1, verbose=1)


Epoch 1/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6669 - loss: 0.6283 - val_accuracy: 0.6985 - val_loss: 0.5923
Epoch 2/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 968us/step - accuracy: 0.6975 - loss: 0.5914 - val_accuracy: 0.7046 - val_loss: 0.5819
Epoch 3/3
282/282 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7031 - loss: 0.5829 - val_accuracy: 0.7067 - val_loss: 0.5736


Évaluation du modèle 2.

In [ ]:
y_pred2 = (model2.predict(X_test_pad) > 0.5).astype(int).ravel()
print('Model 2 Accuracy:', accuracy_score(y_test, y_pred2))
print('Model 2 F1:', f1_score(y_test, y_pred2))


625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 437us/step
Model 2 Accuracy: 0.7024
Model 2 F1: 0.7096868598185543


## 7) Test rapide BERT
Je fais un test très simple avec pipeline Transformers (juste pour voir).

In [ ]:
from transformers import pipeline
sample_texts = X_test.sample(20, random_state=42).tolist()
clf = pipeline('sentiment-analysis')
preds = clf(sample_texts)
preds[:5]


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9854293465614319},
 {'label': 'NEGATIVE', 'score': 0.9856821894645691},
 {'label': 'NEGATIVE', 'score': 0.9958646297454834},
 {'label': 'NEGATIVE', 'score': 0.8398902416229248},
 {'label': 'NEGATIVE', 'score': 0.9622980356216431}]

---
**Notes perso :**
- Le modèle 1 (embedding appris) marche mieux que le Word2Vec pour le moment.
- BERT est juste testé rapidement ici, pas entraîné.